In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import mne

# === Paths ===
TRAIN_CSV = "C:/Users/Kevin Tran/Documents/Project Data/train.csv"
EEG_DIR = "C:/Users/Kevin Tran/Documents/GitHub ED1/hms-harmful-brain-activity-classification/train_eegs"
OUTPUT_FEATURES = "C:/Users/Kevin Tran/Documents/Project Data/eeg_features_labeled.csv"

# === Sampling Rate ===
FS = 200  # Hz

# === EEG Frequency Bands ===
BANDS = {
    "delta": (1, 3), "theta": (4, 7), "alpha1": (8, 9), "alpha2": (10, 12),
    "beta1": (13, 17), "beta2": (18, 30), "gamma1": (31, 40), "gamma2": (41, 50), "higher": (51, 100)
}

# === Preprocessing ===
def apply_notch_filter(signal, fs=FS, freq=60.0, quality_factor=30):
    b, a = iirnotch(w0=freq, Q=quality_factor, fs=fs)
    return filtfilt(b, a, signal)

def apply_bandpass_filter(signal, fs=FS, lowcut=0.5, highcut=40.0, order=5):
    nyquist = 0.5 * fs
    low, high = lowcut / nyquist, highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def normalize_signal(signal):
    return (signal - np.mean(signal)) / np.std(signal)

def apply_ica(signal_df, fs=FS, n_components=10):
    try:
        n_components = min(n_components, len(signal_df.columns))
        info = mne.create_info(ch_names=list(signal_df.columns), sfreq=fs, ch_types=["eeg"] * len(signal_df.columns))
        raw = mne.io.RawArray(signal_df.values.T, info)
        raw.filter(l_freq=1.0, h_freq=None)
        ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter="auto")
        ica.fit(raw)
        raw_clean = ica.apply(raw)
        return pd.DataFrame(raw_clean.get_data().T, columns=signal_df.columns)
    except:
        return signal_df

# === Feature Extraction ===
def extract_time_features(signal):
    from scipy.stats import skew, kurtosis
    return {
        "mean": np.mean(signal),
        "variance": np.var(signal),
        "skewness": skew(signal),
        "kurtosis": kurtosis(signal),
        "rms": np.sqrt(np.mean(signal**2)),
        "zero_crossing_rate": np.sum(np.diff(np.sign(signal)) != 0) / len(signal),
        "mean_abs": np.mean(np.abs(signal)),
        "diff_rms1": np.sqrt(np.mean(np.diff(signal) ** 2)),
        "diff_rms2": np.sqrt(np.mean(np.diff(signal, n=2) ** 2))
    }

def extract_frequency_features(signal, fs=FS):
    L = len(signal)
    Y = np.fft.fft(signal)
    P2 = np.abs(Y / L)
    P1 = P2[:L // 2 + 1]
    P1[1:-1] *= 2
    freqs = fs * np.arange(L // 2 + 1) / L
    band_power = {band: np.sum(P1[(freqs >= low) & (freqs <= high)]) for band, (low, high) in BANDS.items()}
    band_power["spectral_entropy"] = -np.sum(P1 * np.log(P1 + 1e-10))
    return band_power

def extract_features(segment_df):
    all_feats = []
    for channel in segment_df.columns:
        signal = segment_df[channel].values
        time_feats = extract_time_features(signal)
        freq_feats = extract_frequency_features(signal)
        channel_feats = {"channel": channel, **time_feats, **freq_feats}
        all_feats.append(channel_feats)
    df = pd.DataFrame(all_feats)
    return df.groupby("channel").mean().reset_index()

# === Main Processing ===
train_df = pd.read_csv(TRAIN_CSV)
features_list = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extracting labeled EEG segments"):
    eeg_id = row['eeg_id']
    label = row['expert_consensus']
    offset = row['eeg_label_offset_seconds']

    eeg_path = os.path.join(EEG_DIR, f"{eeg_id}.parquet")
    if not os.path.exists(eeg_path):
        continue

    try:
        df = pd.read_parquet(eeg_path)
        start = int((offset + 20) * FS)
        end = start + 10 * FS
        segment = df.iloc[start:end].copy()

        # Preprocess
        for ch in segment.columns:
            segment[ch] = normalize_signal(apply_bandpass_filter(apply_notch_filter(segment[ch].values)))
        segment = apply_ica(segment)

        # Extract features
        feats = extract_features(segment)
        feats['eeg_id'] = eeg_id
        feats['label'] = label
        features_list.append(feats)
    
    except Exception as e:
        print(f"Failed on {eeg_id}: {e}")

# === Combine and Save ===
final_df = pd.concat(features_list, ignore_index=True)
final_df.to_csv(OUTPUT_FEATURES, index=False)
print(f"✅ Saved extracted features to {OUTPUT_FEATURES}")
